In [20]:
from typing import List, TypedDict
import re

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel

from langgraph.graph import StateGraph, END, START
from dotenv import load_dotenv

load_dotenv()

False

In [21]:
docs = PyPDFLoader("../documents/MilanRaiMERN.pdf").load()

In [22]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-11-19T02:50:25+00:00', 'author': '', 'keywords': '', 'moddate': '2026-11-19T02:50:25+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../documents/MilanRaiMERN.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='MILAN RAI\nMERN Stack Developer\nMorang, Nepal | +977-9823459115 | milansampang14@gmail.com | linkedin.com/in/milanrai14 | github.com/milanrai14\nPROFESSIONAL SUMMARY\nMERN Stack Developer focused on building full-stack web applications with React.js, Node.js, Express.js, TypeScript, and\nMongoDB. Experienced in designing RESTful APIs, integrating frontend and backend services, and developing database-\ndriven applications, with additional experience in Python-based AI development using FastAPI, LangChain, and Ollama.\nInterested i

In [23]:
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50).split_documents(docs)

In [24]:
for d in chunks: 
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [25]:
embeddings = OllamaEmbeddings(model="bge-m3")

In [26]:
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="../haha_db"
)

In [27]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={'k': 4})

In [28]:
llm = ChatOllama(model="llama3.2")

In [29]:
class State(TypedDict): 
    question: str 
    docs: List[Document]

    strips: List[str]
    kept_strips: List[str]
    refined_context: str

    answer: str

In [30]:
def retrieve(state: State) -> State: 
    q = state['question']
    return{'docs': retriever.invoke(q)}

In [31]:
# Sentence level decompose
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 20]

In [32]:
#Filter (LLM judge)
class KeepOrDrop(BaseModel): 
    keep: bool
    

In [34]:
filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "You are a strict relevance filter. \n"
            "Return keep=true only if the sentence directly helps answer the question. \n"
            "Use ONLY the sentence. Output JSON only."
        ),
        ("human", "Question: {question}\n\nSentence:\n{sentence}")
    ]
)

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

In [37]:
# Refining (Decompose -> Filter -> Recompose)
def refine(state: State) -> State: 
    q = state['question']

    #Combine retrieved docs into one context string
    context = "\n\n".join(d.page_content for d in state['docs']).strip()

    #Decomposition: context -> sentence strips
    strips = decompose_to_sentences(context)

    #Filter: keeps only relevant strips
    kept: List[str] = []

    for s in strips: 
        if filter_chain.invoke({"question": q, "sentence": s}).keep:
            kept.append(s)
    #Recompose: glue kept strips back together (internal knowledge)
    refined_context = "\n".join(kept).strip()

    return{
        "strips": strips,
        "kept_strips": kept, 
        "refined_context": refined_context
    }